In [11]:
import pandas as pd, os, textgrids, numpy as np
import scipy.io.wavfile as wavfile
from scipy import signal as sgn

# Mistakes folder
mistake_folder = os.path.normpath(path='C:/Users/User/repos/Speech-encoding/Datos/mistakes/')
data_info = pd.read_csv(filepath_or_buffer=os.path.join(mistake_folder, 'Filtrado_anotaciones.csv'), header=0, delimiter=';')

# La sesion con más errores es la 25 --> 48 errores, 27 en canal 1 y 21 en canal 2. La señal del que habla, pero podemos implementarlo con las 4 variantes
filtered_mistake_folder = os.path.join(mistake_folder,'Filtrados/')

# # Rename filepath in order to include the channel
# for file in [f for f in os.listdir(filtered_mistake_folder) if f.endswith('.TextGrid')]:
#     # Get the filename and open it to extract de chanel number
#     textgrids_path = os.path.normpath(os.path.join(filtered_mistake_folder, file))
#     grid = textgrids.TextGrid(textgrids_path)
#     channel_number = int(grid[list(grid.keys())[0]][0].text.split('Canal: ')[1][0])
    
#     # Define renamed file
#     session = int(file.split('s')[1][:2])
#     obj = int(file.split('_')[2][:2])
    # new_name_file = f'filtered_session{session}_object{obj}_channel{channel_number}.TextGrid'

    # # Rename the file to include this information in its name
    # os.rename(textgrids_path, os.path.normpath(os.path.join(filtered_mistake_folder, new_name_file)))

In [13]:
# Read file
wav = wavfile.read(r'Datos\wavs\S21\s21.objects.01.channel1.wav')[1]
wav = wav.astype("float")
envelope = np.abs(sgn.hilbert(wav))
window_size, stride = int(16e3/128), int(16e3/128)
envelope = np.array([np.mean(envelope[i:i+window_size]) for i in range(0, len(envelope), stride) if i+window_size<=len(envelope)])
envelope =  envelope.reshape(-1, 1)

In [15]:
envelope.shape

(9168, 1)

In [38]:
filtered_mistakes = pd.read_csv(r'Datos\mistakes\filtrado_anotaciones.csv', delimiter=';')
# filtered_mistakes[['Etiqueta Error AT', 'Etiqueta Error GM']]'palabra seleccionada', 'palabra previa','palabra siguiente', 'frase/contexto', 'Posicion del error en la frase',

mistake_start = float(filtered_mistakes['palabra seleccionada'][0].split(',')[0].replace('(',''))
mistake_end = float(filtered_mistakes['palabra seleccionada'][0].split(',')[1])

phrases = pd.read_table('Datos/phrases/S21/s21.objects.01.channel1.phrases', header=None, sep="\t")
start_time, end_time = phrases[0].iloc[0], phrases[1].iloc[-1]
phrases_time = np.arange(start_time, end_time, 1/128)

onset_filter = mistake_start<=phrases_time
offset_filter = phrases_time<=mistake_end
mistake_signal = np.zeros(shape=phrases_time.shape)
mistake_signal[onset_filter&offset_filter] = np.ones(shape=np.sum(onset_filter&offset_filter))



In [80]:
filtered_mistakes['Etiqueta del error']

0      A
1      L
2      A
3      L
4      D
      ..
200    L
201    A
202    A
203    A
204    A
Name: Etiqueta del error, Length: 205, dtype: object

In [83]:
canal = filtered_mistakes['filename/canal']

0      ('s26_objects_10.TextGrid', '2')
1      ('s26_objects_05.TextGrid', '2')
2      ('s26_objects_05.TextGrid', '2')
3      ('s24_objects_06.TextGrid', '2')
4      ('s24_objects_01.TextGrid', '1')
                     ...               
200    ('s26_objects_23.TextGrid', '2')
201    ('s25_objects_09.TextGrid', '1')
202    ('s26_objects_24.TextGrid', '2')
203    ('s26_objects_24.TextGrid', '2')
204    ('s29_objects_25.TextGrid', '1')
Name: filename/canal, Length: 205, dtype: object